# 01 · Policies, retrieval and access control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/hr-tech/hr-policy-rag-assistant/notebooks/01_corpus_retrieval_and_access.ipynb)

The assistant answers employees' HR questions from the company's own policies, and from nothing else.
The policies here belong to a fictional car maker, Northlake Motors GmbH: 13 documents on leave,
sickness, shifts, travel, parental leave and more, written for this project. Two of them are
restricted: one for managers and one for HR only.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "hr-tech/hr-policy-rag-assistant"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "hrrag").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: hr-policy-rag-assistant


In [2]:
from hrrag.config import Settings
from hrrag.corpus import load
from hrrag.retrieval import Retriever, visible
from hrrag.answer import Assistant

passages = load(Path("policies"))
retriever = Retriever(passages, "hybrid")
assistant = Assistant(retriever, Settings())
print(f"{len(passages)} passages from {len({p.policy_id for p in passages})} policies")

52 passages from 13 policies


## From documents to passages

Each policy is a Markdown file with a header: id, version, effective date, audience and owner.
Passages follow the document's own sections, so a citation points to a place a person can open
and check.

In [3]:
pd.DataFrame([{"pid": p.pid, "audience": p.audience, "version": p.version, "words": len(p.text.split()), "text": p.text[:70] + "…"}
              for p in passages]).head(10)

,pid,audience,version,words,text
0,HR-01#entitlement,employee,3.2,64,Employees on a five-day week receive 28 working days of paid annual le…
1,HR-01#requesting-leave,employee,3.2,57,Leave is requested in the HR portal. Requests for more than five conse…
2,HR-01#carry-over,employee,3.2,59,Leave that cannot be taken in the calendar year may be carried over un…
3,HR-01#special-leave,employee,3.2,64,Employees receive paid special leave for the following events: their o…
4,HR-09#shuttle-buses,employee,2.2,27,"Free shuttle buses run between Erkner station and the site, timed to t…"
5,HR-09#public-transport-ticket,employee,2.2,27,The company subsidises the Deutschlandticket with 30 euros per month. …
6,HR-09#bike-leasing,employee,2.2,31,Employees can lease up to two bikes or e-bikes through salary conversi…
7,HR-09#parking,employee,2.2,24,Parking at the site is free. Charging points for electric cars are ava…
8,HR-13#salary-bands,hr,2026.1,49,Salary bands are defined per job level from L1 to L8. Every band has a…
9,HR-13#merit-budget,hr,2026.1,39,The merit increase budget for 2026 is 3.5 percent of the base salary s…


In [4]:
pd.Series([p.audience for p in passages]).value_counts().rename("passages")

employee    45
manager      4
hr           3
Name: passages, dtype: int64

## Access control happens before ranking

Passages the role may not read are removed before anything is scored, so they cannot leak
through a score, a snippet or a citation. The same question gets different results for an
employee and for HR.

In [5]:
q = "What is the merit budget for 2026?"
for role in ("employee", "hr"):
    hits, conf = retriever.search(q, role=role, k=3)
    print(f"{role:>8}  confidence {conf:.2f}  ", [h.passage.pid for h in hits])

employee  confidence 0.16   ['HR-08#individual-budget', 'HR-08#german-language-courses', 'HR-10#what-we-process']
      hr  confidence 0.42   ['HR-13#merit-budget', 'HR-13#pay-equity-review', 'HR-08#individual-budget']


## Questions in employees' own words

People ask "how much Urlaub", "doctor's note" or "home office", while the policies say "annual
leave", "medical certificate" and "mobile work". A synonym map adds the policies' own words to the
query, and character n-grams absorb typos and word forms.

In [6]:
from hrrag.text import expand
for q in ["How much Urlaub do I have?", "Do I need a Krankmeldung on day 2?", "Can I do home office from Spain?"]:
    print(q, "->", expand(q))

How much Urlaub do I have? -> How much Urlaub do I have? entitlement receive working days annual leave
Do I need a Krankmeldung on day 2? -> Do I need a Krankmeldung on day 2? sick certificate
Can I do home office from Spain? -> Can I do home office from Spain? remote mobile


In [7]:
for q in ["How much Urlaub do I have?", "Do I need a doctors note on day 2?", "night shift Zuschlag", "Can I work from Spain for a few weeks?"]:
    a = assistant.ask(q)
    src = f"{a.citations[0]['policy_id']}, {a.citations[0]['section']}" if a.citations else "none (declined)"
    print(f"Q: {q}\nA: {a.text}\n   source: {src}, confidence {a.confidence:.2f}\n")

Q: How much Urlaub do I have?
A: Employees on a five-day week receive 28 working days of paid annual leave per calendar year. [1] In the year an employee joins or leaves, the entitlement is one twelfth of the annual amount for each full month of employment. [1]
   source: HR-01, Entitlement, confidence 0.48

Q: Do I need a doctors note on day 2?
A: A medical certificate of incapacity for work is required from the fourth calendar day of illness. [1] Doctors in Germany transmit the certificate electronically to the health insurer, and HR retrieves it; you only need to tell your manager that you saw a doctor. [1]
   source: HR-02, Medical certificate, confidence 0.23

Q: night shift Zuschlag
A: Night, Sunday and public holiday allowances are tax-free within the statutory limits; the payslip shows them separately. [1]
   source: HR-04, Payment and tax, confidence 0.28

Q: Can I work from Spain for a few weeks?
A: I could not find this in the HR policies I have access to. Please ask your HR

The first two are right. The last two show the limits, and they are left in on purpose. The
Zuschlag question lands on the section about how allowances are taxed instead of the one with the
rates, because both mention night work. The question about Spain scores 0.167, just under the 0.17
threshold, so the assistant declines rather than guessing. Notebook 02 counts misses like these
systematically.